# 3D Shogi AI Training in Google Colab
このノートブックを実行して、Google Colab 上で 3D将棋AI の学習を回します。Drive の `ai_training` フォルダだけで動く構成です。

## 1. Drive をマウント

In [ ]:
# 1. Google Drive のマウント
from google.colab import drive
drive.mount('/content/drive')

## 2. プロジェクトを Colab 作業ディレクトリへコピー

In [ ]:
# 2. ai_training フォルダを Colab 作業ディレクトリへコピー
# 置き場所: Google Drive の MyDrive 直下に ai_training フォルダ
# 例: /content/drive/MyDrive/ai_training
import os
import shutil

src_candidates = [
    "/content/drive/MyDrive/ai_training",
    "/content/drive/MyDrive/3d_shogi/ai_training",
]

src = None
for p in src_candidates:
    if os.path.isdir(p):
        src = p
        break

if src is None:
    print("ai_training フォルダが見つかりません。")
    print("次のどちらかに置いてください:")
    for p in src_candidates:
        print(" -", p)
    print("Drive直下の一覧:")
    print(os.listdir("/content/drive/MyDrive")[:50])
    raise SystemExit(1)

if os.path.exists("/content/ai_training"):
    shutil.rmtree("/content/ai_training")

shutil.copytree(src, "/content/ai_training")
print("copied from:", src)
%cd /content/ai_training
!ls -la | head -n 30

## 3. 依存ライブラリをインストール

In [ ]:
# 3. 依存ライブラリのインストール
%cd /content/ai_training
!python -m pip install --upgrade pip
!python -m pip install -r requirements_colab.txt

## 4. 学習スクリプトを実行（新規/自動再開）
既存モデルがあれば自動ロードされます。

In [ ]:
# 4. 学習スタート（新規/自動再開）
# /content/drive/MyDrive/3d_shogi_train/models に最新モデルがあれば自動ロード
%cd /content/ai_training
!python train_colab.py --drive-root /content/drive/MyDrive/3d_shogi_train --n-envs 2 --n-steps 4096 --n-epochs 4 --batch-size 256 --max-mask-gb 8 --device cuda --force-cuda --fixed-action-space

## 5. TensorBoard（任意）

In [ ]:
# 6. TensorBoard（イベントファイルを自動探索して起動）
import glob
import os

candidates = [
    "/content/drive/MyDrive/3d_shogi_train/logs",
    "/content/ai_training/logs",
    "/content/drive/MyDrive/ai_training/logs",
]

print("TensorBoard log candidates:")
for d in candidates:
    exists = os.path.isdir(d)
    n_events = len(glob.glob(os.path.join(d, "**", "events.out.tfevents.*"), recursive=True)) if exists else 0
    print(f"- {d} | exists={exists} | events={n_events}")

logdir = None
for d in candidates:
    if os.path.isdir(d) and glob.glob(os.path.join(d, "**", "events.out.tfevents.*"), recursive=True):
        logdir = d
        break

if logdir is None:
    raise RuntimeError("イベントファイルが見つかりません。先に学習セルを実行し、1世代以上進めてから再実行してください。")

print("Using logdir:", logdir)
%load_ext tensorboard
%tensorboard --logdir $logdir

## 6. アップロード必須ファイルの確認

In [ ]:
# 6. 必須ファイル確認
!ls -la /content/drive/MyDrive/ai_training
!test -f /content/drive/MyDrive/ai_training/train_colab.py && echo "OK: train_colab.py"
!test -f /content/drive/MyDrive/ai_training/train.py && echo "OK: train.py"
!test -f /content/drive/MyDrive/ai_training/shogi_env.py && echo "OK: shogi_env.py"
!test -f /content/drive/MyDrive/ai_training/model.py && echo "OK: model.py"

## 7. 再実行セル（継続学習）
同じ `--drive-root` を使えば最新モデルから継続します。

In [ ]:
# 7. 継続学習セル（同じ drive-root で再実行）
%cd /content/ai_training
!python train_colab.py --drive-root /content/drive/MyDrive/3d_shogi_train --n-envs 2 --n-steps 4096 --n-epochs 4 --batch-size 256 --max-mask-gb 8 --device cuda --force-cuda --fixed-action-space